In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Carregar os datasets
df_train = pd.read_csv(f'challenge-webmedia-e-globo-2023/files/treino/treino_parte1.csv')
df_items = pd.read_csv(f'challenge-webmedia-e-globo-2023/itens/itens/itens-parte1.csv')

In [2]:
df_train.head()

,userId,userType,historySize,history,timestampHistory,numberOfClicksHistory,timeOnPageHistory,scrollPercentageHistory,pageVisitsCountHistory,timestampHistory_new
0,f98d1132f60d46883ce49583257104d15ce723b3bbda21...,Non-Logged,3,"c8aab885-433d-4e46-8066-479f40ba7fb2, 68d2039c...","1657146417045, 1657146605778, 1657146698738","76, 38, 41","20380, 21184, 35438","50.3, 18.18, 16.46","2, 1, 1","1657146417045, 1657146605778, 1657146698738"
1,2c1080975e257ed630e26679edbe4d5c850c65f3e09f65...,Non-Logged,60,"3325b5a1-979a-4cb3-82b6-63905c9edbe8, fe856057...","1656684240278, 1656761266729, 1656761528085, 1...","7, 80, 2, 1, 7, 62, 26, 44, 4, 4, 14, 45, 13, ...","6049, 210489, 8672, 10000, 30000, 123007, 9965...","25.35, 45.66, 35.3, 28.05, 36.53, 47.57, 55.33...","1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 2, 1, 1...","1656684240278, 1656761266729, 1656761528085, 1..."
2,0adffd7450d3b9840d8c6215f0569ad942e782fb19b805...,Logged,107,"04756569-593e-4133-a95a-83d35d43dbbd, 29b6b142...","1656678946256, 1656701076495, 1656701882565, 1...","0, 0, 0, 0, 0, 44, 0, 0, 2, 1, 0, 0, 0, 44, 0,...","311274, 140000, 32515, 157018, 118689, 159243,...","67.58, 47.22, 41.52, 63.09, 51.38, 65.11, 71.9...","1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1...","1656678946256, 1656701076495, 1656701882565, 1..."
3,c1e8d644329a78ea1f994292db624c57980b2886cfbc2d...,Non-Logged,56,"1f2b9c2f-a2d2-4192-b009-09065da8ec23, 04756569...","1658333312180, 1658404553818, 1658408449062, 1...","8, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 2, 0, 1, 1...","182696, 91925, 30000, 273655, 126409, 42980, 1...","58.26, 72.66, 22.57, 59.89, 40.36, 36.35, 14.7...","1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1...","1658333312180, 1658404553818, 1658408449062, 1..."
4,e777d1f31d4d955b63d60acc13df336d3903f52ab8f8f4...,Non-Logged,4,"bebdeb3e-1699-43e0-a1b8-989f5a6ab679, f4b484a7...","1658766608801, 1658766608801, 1660084035094, 1...","579, 579, 7, 2","801396, 801396, 10000, 10000","78.74, 78.74, 16.71, 9.34","7, 7, 1, 1","1658766608801, 1658766608801, 1660084035094, 1..."


In [3]:
df_items.head()

,page,url,issued,modified,title,body,caption
0,13db0ab1-eea2-4603-84c4-f40a876c7400,http://g1.globo.com/am/amazonas/noticia/2022/0...,2022-06-18 20:37:45+00:00,2023-04-15 00:02:08+00:00,Caso Bruno e Dom: 3º suspeito tem prisão tempo...,"Após audiência de custódia, a Justiça do Amazo...",Jeferson da Silva Lima foi escoltado por agent...
1,92907b73-5cd3-4184-8d8c-e206aed2bf1c,http://g1.globo.com/pa/santarem-regiao/noticia...,2019-06-20 17:19:52+00:00,2023-06-16 20:19:15+00:00,Linguajar dos santarenos é diferenciado e chei...,Vista aérea de Santarém\nÁdrio Denner/ AD Prod...,As expressões santarenas não significam apenas...
2,61e07f64-cddf-46f2-b50c-ea0a39c22050,http://g1.globo.com/mundo/noticia/2022/07/08/e...,2022-07-08 08:55:52+00:00,2023-04-15 04:25:39+00:00,Ex-premiê Shinzo Abe morre após ser baleado no...,Novo vídeo mostra que assassino de Shinzo Abe ...,Ex-primeiro-ministro foi atingido por tiros de...
3,30e2e6c5-554a-48ed-a35f-6c6691c8ac9b,http://g1.globo.com/politica/noticia/2021/09/0...,2021-09-09 19:06:46+00:00,2023-06-07 17:44:54+00:00,"Relator no STF, Fachin vota contra marco tempo...","Relator no STF, Fachin vota contra marco tempo...",Ministro defendeu que posse indígena é diferen...
4,9dff71eb-b681-40c7-ac8d-68017ac36675,http://g1.globo.com/politica/noticia/2021/09/1...,2021-09-15 19:16:13+00:00,2023-06-07 17:43:39+00:00,"\nApós 2 votos, pedido de vista suspende julga...",Após um pedido de vista (mais tempo para análi...,"Pelo marco temporal, índios só podem reivindic..."


In [4]:
df_train["history"][0].split(", ")

['c8aab885-433d-4e46-8066-479f40ba7fb2',
 '68d2039c-c9aa-456c-ac33-9b2e8677fba7',
 '13e423ce-1d69-4c78-bc18-e8c8f7271964']

In [5]:
teste = df_train["history"].str.split(", ")
teste.explode().value_counts()

history
d2593c3d-2347-40d9-948c-b6065e8459a9    4282
f6b5d170-48b9-4f8e-88d4-c84b6668f3bd    3873
1f32787b-de2b-49be-8c20-ddaeae34cc22    3395
f0a78e58-ec7e-494c-9462-fbd6446a9a89    3192
6a83890a-d9e9-4f6b-a6c6-90d031785bbf    3121
                                        ... 
22c36dd5-ef5e-4326-948e-b2fe7d10ed26       1
286b2ad0-dfa6-496f-a9ae-f773b9782427       1
3137dbaf-d683-44b1-b512-222287710829       1
15fd7d1f-1081-4b22-b50d-b508273607d5       1
b8ced8b2-c82b-42da-bfa9-2ac642127030       1
Name: count, Length: 108573, dtype: int64

In [6]:
# Criar um ranking de popularidade baseado no número de cliques
popular_articles = df_train["history"].str.split(", ").explode().value_counts().index.tolist()
popular_articles

['d2593c3d-2347-40d9-948c-b6065e8459a9',
 'f6b5d170-48b9-4f8e-88d4-c84b6668f3bd',
 '1f32787b-de2b-49be-8c20-ddaeae34cc22',
 'f0a78e58-ec7e-494c-9462-fbd6446a9a89',
 '6a83890a-d9e9-4f6b-a6c6-90d031785bbf',
 'bf257382-74fb-4392-ad6a-143240e39f81',
 '855d20b7-53f2-4678-a10f-55402d085018',
 '4c63d7cd-4902-4ffb-9b94-578b1b2151f0',
 '1c27cf97-b20c-4e40-b1f1-288b721517b3',
 'a36c98b5-f159-48f8-9f5a-1fc6ea9956c8',
 '4e9c2825-ff13-41ca-8e91-edd848060d19',
 '7b056bf6-c232-46bd-8903-59145ff7ce46',
 '29b6b142-4173-4ec4-832f-7d0a32255c10',
 '89fa73f0-4341-4de4-bb2a-e429ef96bd43',
 'e384ec29-136e-4241-9321-49b367b8cbd5',
 '882e7c95-935a-4eab-9ece-f85f5f7d0f4e',
 '15281e10-e6bc-48bc-9b1b-94402f83699b',
 '4700f517-5c5d-483c-81b7-d77aca04991c',
 '8c246d2b-81bd-4c1f-b563-2c905675f984',
 '458bf0ec-efb4-4bfd-9446-c80295e6aa87',
 '61e07f64-cddf-46f2-b50c-ea0a39c22050',
 '5dff8fb2-73e6-4c22-a34f-c367aa2677df',
 'e5185368-70f8-4998-a738-ca22f300da7b',
 'a6ab18ec-e32f-474b-964b-987309c61581',
 'bd4e7054-4043-

In [7]:
# Criar embeddings das notícias usando TF-IDF
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

stop = stopwords.words("portuguese")
stop


[nltk_data] Downloading package stopwords to
[nltk_data]     /home/felipelangoniramos/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


['a',
 'à',
 'ao',
 'aos',
 'aquela',
 'aquelas',
 'aquele',
 'aqueles',
 'aquilo',
 'as',
 'às',
 'até',
 'com',
 'como',
 'da',
 'das',
 'de',
 'dela',
 'delas',
 'dele',
 'deles',
 'depois',
 'do',
 'dos',
 'e',
 'é',
 'ela',
 'elas',
 'ele',
 'eles',
 'em',
 'entre',
 'era',
 'eram',
 'éramos',
 'essa',
 'essas',
 'esse',
 'esses',
 'esta',
 'está',
 'estamos',
 'estão',
 'estar',
 'estas',
 'estava',
 'estavam',
 'estávamos',
 'este',
 'esteja',
 'estejam',
 'estejamos',
 'estes',
 'esteve',
 'estive',
 'estivemos',
 'estiver',
 'estivera',
 'estiveram',
 'estivéramos',
 'estiverem',
 'estivermos',
 'estivesse',
 'estivessem',
 'estivéssemos',
 'estou',
 'eu',
 'foi',
 'fomos',
 'for',
 'fora',
 'foram',
 'fôramos',
 'forem',
 'formos',
 'fosse',
 'fossem',
 'fôssemos',
 'fui',
 'há',
 'haja',
 'hajam',
 'hajamos',
 'hão',
 'havemos',
 'haver',
 'hei',
 'houve',
 'houvemos',
 'houver',
 'houvera',
 'houverá',
 'houveram',
 'houvéramos',
 'houverão',
 'houverei',
 'houverem',
 'hou

In [8]:
vectorizer = TfidfVectorizer(stop_words=stop)
tfidf_matrix = vectorizer.fit_transform(df_items['title'] + " " + df_items['body'])

In [9]:
# Criar um dicionário para mapear páginas aos índices
doc_indices = {page: idx for idx, page in enumerate(df_items['page'])}
doc_indices

{'13db0ab1-eea2-4603-84c4-f40a876c7400': 0,
 '92907b73-5cd3-4184-8d8c-e206aed2bf1c': 1,
 '61e07f64-cddf-46f2-b50c-ea0a39c22050': 2,
 '30e2e6c5-554a-48ed-a35f-6c6691c8ac9b': 3,
 '9dff71eb-b681-40c7-ac8d-68017ac36675': 4,
 'a9fd6d34-6f40-4c90-849b-2ad36f04fd6f': 5,
 'c2893f07-08fe-4b4c-8904-92f0fedc3040': 6,
 '682da2fa-6f5b-4017-be35-7968990f62b9': 7,
 'ae5a4574-9347-40c4-b17a-bc20f13faa67': 8,
 '360c9076-0302-47dc-a4fe-da301d151aa2': 9,
 '7d49bc26-de32-45ac-98e9-e66c3359dc72': 10,
 'da15522f-f679-4ff4-842c-9a212d5af519': 11,
 '790f8f6d-2674-462d-ab94-47239c2f60d5': 12,
 'edc93569-bf37-437d-a6c3-fda029f8bc70': 13,
 'e5323b08-013b-408d-91c5-9c03460b689d': 14,
 '99ab59ae-c364-4657-93a6-6a23b58a1a51': 15,
 '71cdc0aa-902a-4b82-a518-c6e4411b5fac': 16,
 'dc4aea98-84c9-47cf-a173-cb58820d5f28': 17,
 'b88ba573-3cbb-4595-9b92-1194bdda9d44': 18,
 'eaa6c7e1-38ba-4d9c-9570-2618df4297f2': 19,
 'e0940638-1bf2-43b2-99c7-d970783463a4': 20,
 '6e9150e5-57f8-4762-b5ea-d82164c91788': 21,
 '6eafb73e-e0be-4f89

In [10]:
doc_indices["13db0ab1-eea2-4603-84c4-f40a876c7400"]

0

In [42]:
def recommend_news(user_history, top_n=5):
    if not user_history:  # Cold start (usuário novo)
        return df_items[df_items['page'].isin(popular_articles[:top_n])][['page', 'title']]
    
    # Obter índices das notícias que o usuário viu
    viewed_indices = [doc_indices[page] for page in user_history if page in doc_indices]
    if not viewed_indices:
        return df_items[df_items['page'].isin(popular_articles[:top_n])][['page', 'title']]
    
    # Calcular similaridade entre as notícias vistas e todas as outras
    user_profile = np.mean(tfidf_matrix[viewed_indices], axis=0)
    scores = cosine_similarity(user_profile, tfidf_matrix).flatten()
    
    # Obter as top-N notícias mais similares que o usuário ainda não viu
    recommended_indices = np.argsort(scores)[::-1]
    recommended_pages = [df_items.iloc[i]['page'] for i in recommended_indices if df_items.iloc[i]['page'] not in user_history]
    history_pages = [df_items.iloc[i]['page'] for i in viewed_indices if df_items.iloc[i]['page'] not in user_history]
    return (df_items[df_items['page'].isin(recommended_pages[:top_n])][['page', 'title']] ,df_items[df_items['page'].isin(history_pages[:top_n])][['page', 'title']])

# Testar com um usuário aleatório
sample_user_history = df_train.iloc[0]['history'].split(", ")
sample_user_history

['c8aab885-433d-4e46-8066-479f40ba7fb2',
 '68d2039c-c9aa-456c-ac33-9b2e8677fba7',
 '13e423ce-1d69-4c78-bc18-e8c8f7271964']

In [43]:
recommendations = recommen_news(sample_user_history, top_n=5)
recommendations

,page,title
15859,f0a78e58-ec7e-494c-9462-fbd6446a9a89,Caso Bárbara: suspeito de envolvimento no assa...
23198,d2593c3d-2347-40d9-948c-b6065e8459a9,Anestesista é preso em flagrante por estupro d...
26314,6a83890a-d9e9-4f6b-a6c6-90d031785bbf,Pizzaria recebe PIX falso e entrega refrigeran...
60121,1f32787b-de2b-49be-8c20-ddaeae34cc22,Filha é presa por golpe estimado em R$ 725 mil...
71660,f6b5d170-48b9-4f8e-88d4-c84b6668f3bd,Diretor da Caixa Econômica Federal é encontrad...


In [13]:
import pickle
import joblib

# Salvar TF-IDF Vectorizer
with open("tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

# Salvar Matriz TF-IDF
joblib.dump(tfidf_matrix, "tfidf_matrix.pkl")

# Salvar Dicionário de Índices
with open("doc_indices.pkl", "wb") as f:
    pickle.dump(doc_indices, f)
